# Remote setup (Google Colab)

This notebook prepares a remote Colab checkout and runs the locked data workflow.

Setup runs only in Google Colab. Outside Colab, executing this notebook skips setup.

`uv sync` does not configure Colab's kernel. For interactive imports, use `01_data_exploration.ipynb` locally. Scripts run in uv subprocesses.

Example commands:
```bash
!git clone --branch main https://github.com/rbl-bidilatimotei/sleep-classifier.git
!uv run --locked --group notebook python -m sleep_classifier.data download --config configs/explore.yaml
!uv run --locked --group notebook python -m sleep_classifier.data audit --config configs/explore.yaml
!uv run --locked --group notebook python scripts/execute_notebook.py
```

Set `BRANCH_OR_REF` to a branch or tag before running. That ref must contain setup files, including `uv.lock` and `configs/explore.yaml`.

If `/content/sleep-classifier` exists, the cell will not clone over it, switch branches, pull, or reset it. Its current commit must match the selected ref.

In [ ]:
import os
import subprocess
import sys
from importlib.util import find_spec
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    print("Colab setup skipped: this notebook only runs setup in Google Colab.")
else:
    REPOSITORY_URL = "https://github.com/rbl-bidilatimotei/sleep-classifier.git"
    BRANCH_OR_REF = "main"
    REPOSITORY_DIR = Path("/content/sleep-classifier")

    def run_checked(
        command: list[str | Path],
        *,
        cwd: Path | None = None,
        capture_output: bool = False,
    ) -> subprocess.CompletedProcess[str]:
        """Run a command and raise when it fails."""
        command_args = list(map(str, command))
        print("$ " + " ".join(command_args))
        try:
            result = subprocess.run(
                command_args,
                cwd=cwd,
                text=True,
                capture_output=capture_output,
                check=True,
            )
        except subprocess.CalledProcessError as error:
            if error.stdout:
                print(error.stdout, end="")
            if error.stderr:
                print(error.stderr, file=sys.stderr, end="")
            raise
        if capture_output:
            if result.stdout:
                print(result.stdout, end="")
            if result.stderr:
                print(result.stderr, file=sys.stderr, end="")
        return result

    if REPOSITORY_DIR.exists():
        if not (REPOSITORY_DIR / ".git").exists():
            raise RuntimeError(
                f"{REPOSITORY_DIR} exists and is not a Git checkout; "
                "move it manually before retrying."
            )
        current_commit = run_checked(
            ["git", "rev-parse", "HEAD"], cwd=REPOSITORY_DIR, capture_output=True
        ).stdout.strip()
        selected_commit = run_checked(
            ["git", "rev-parse", "--verify", f"{BRANCH_OR_REF}^{{commit}}"],
            cwd=REPOSITORY_DIR,
            capture_output=True,
        ).stdout.strip()
        if current_commit != selected_commit:
            raise RuntimeError(
                "Existing checkout does not match BRANCH_OR_REF. Set the variable to its "
                "current branch or tag, or use another directory. No checkout changes were made."
            )
        status = run_checked(
            ["git", "status", "--short"], cwd=REPOSITORY_DIR, capture_output=True
        ).stdout
        print("Using existing checkout at", current_commit)
        if status:
            print("Existing checkout has local changes; they are preserved:\n" + status)
    else:
        REPOSITORY_DIR.parent.mkdir(parents=True, exist_ok=True)
        run_checked(["git", "clone", "--branch", BRANCH_OR_REF, REPOSITORY_URL, REPOSITORY_DIR])

    UV_TARGET = Path("/content/.sleep-classifier-uv")
    if find_spec("uv") is None:
        run_checked(
            [sys.executable, "-m", "pip", "install", "--quiet", "--target", UV_TARGET, "uv"]
        )
        existing_pythonpath = os.environ.get("PYTHONPATH")
        os.environ["PYTHONPATH"] = str(UV_TARGET) + (
            os.pathsep + existing_pythonpath if existing_pythonpath else ""
        )

    UV = [sys.executable, "-m", "uv"]
    run_checked([*UV, "sync", "--locked", "--group", "notebook"], cwd=REPOSITORY_DIR)
    config = REPOSITORY_DIR / "configs" / "explore.yaml"
    run_checked(
        [
            *UV,
            "run",
            "--locked",
            "--group",
            "notebook",
            "python",
            "-m",
            "sleep_classifier.data",
            "download",
            "--config",
            config,
        ],
        cwd=REPOSITORY_DIR,
    )
    run_checked(
        [
            *UV,
            "run",
            "--locked",
            "--group",
            "notebook",
            "python",
            "-m",
            "sleep_classifier.data",
            "audit",
            "--config",
            config,
        ],
        cwd=REPOSITORY_DIR,
    )
    run_checked(
        [*UV, "run", "--locked", "--group", "notebook", "python", "scripts/execute_notebook.py"],
        cwd=REPOSITORY_DIR,
    )

    executed_notebook = (
        REPOSITORY_DIR / "artifacts" / "notebooks" / "01_data_exploration.executed.ipynb"
    )
    html_dir = REPOSITORY_DIR / "artifacts" / "notebooks"
    run_checked(
        [
            *UV,
            "run",
            "--locked",
            "--group",
            "notebook",
            "jupyter",
            "nbconvert",
            "--to",
            "html",
            str(executed_notebook),
            "--output-dir",
            str(html_dir),
        ],
        cwd=REPOSITORY_DIR,
    )
    from IPython.display import HTML, display

    display(HTML(filename=str(html_dir / "01_data_exploration.executed.html")))